In [2]:
# Import packages and initialize Earth Engine

import ee
import geemap
import pandas as pd
import numpy as np
import os
import seaborn as sns

geemap.ee_initialize()


Successfully saved authorization token.


### Initialize variables: Mask, extraction points and time frame 

In [20]:
date_start = ee.Date('2003-08-05') # Earliest station: ZAC
date_end_mod = ee.Date('2020-02-27') # Before orbital drift TERRA
date_end_myd = ee.Date('2021-03-18') # Before orbital drift AQUA

greenlandmask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ocean_mask').eq(0)
greenland = ee.Geometry.Polygon(
[[[-36.29516924635421, 83.70737243835941],
[-51.85180987135421, 82.75597137647488],
[-61.43188799635421, 81.99879137488564],
[-74.08813799635422, 78.10103528196419],
[-70.13305987135422, 75.65372336709613],
[-61.08032549635421, 75.71891096312955],
[-52.20337237135421, 60.9795530382023],
[-43.41430987135421, 58.59235996703347],
[-38.49243487135421, 64.70478286561182],
[-19.771731746354217, 69.72271161037442],
[-15.728762996354217, 76.0828635948066],
[-15.904544246354217, 79.45091003031243],
[-10.015872371354217, 81.62328742628017],
[-26.627200496354217, 83.43179828852398],
[-31.636966121354217, 83.7553561747887]]])

awsPoints = ee.FeatureCollection([

# Kobbefjord
ee.Feature(ee.Geometry.Point([-51.37199020385742, 64.12248229980469]), {"id": 'Kobbefjord_M500'}),

# Disko
# ee.Feature(ee.Geometry.Point([-53.479400634765625, 69.27300262451172]), {"id": 'Disko_T1'}),
# ee.Feature(ee.Geometry.Point([-53.43281936645508, 69.28909301757812]), {"id": 'Disko_T2'}),
# ee.Feature(ee.Geometry.Point([-53.45709991455078, 69.2767105102539]), {"id": 'Disko_T3'}),
# ee.Feature(ee.Geometry.Point([-53.49897003173828, 69.25126647949219]), {"id": 'Disko_T4'}),
ee.Feature(ee.Geometry.Point([-53.514129638671875, 69.25348663330078]), {"id": 'Disko_AWS2'}),

# Zackenberg
ee.Feature(ee.Geometry.Point([-20.563194274902344, 74.46549224853516]), {"id": 'Zackenberg_M2'}),
ee.Feature(ee.Geometry.Point([-20.459354400634766, 74.50310516357422]), {"id": 'Zackenberg_M3'}),
ee.Feature(ee.Geometry.Point([-20.552143096923828, 74.47307586669922]), {"id": 'Zackenberg_M4_30min'})
])


### Create test images 

In [ ]:
# MODIS

def lst(image):
    lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('LST_Day_1km_C')
    lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('LST_Night_1km_C')
    return image.addBands(lst_day).addBands(lst_night)

test_image = ee.ImageCollection('MODIS/061/MOD11A1').filterDate(date_start, date_end_mod).first()
test_image2 = ee.Image('MODIS/061/MOD11A1/2012_06_05')
# test_image = lst(test_image)
test_image2 = lst(test_image2)
# test_image2 = maskQualityDaytime(test_image2)

img_col = (
    ee.ImageCollection('MODIS/061/MOD11A1')
    .filterDate(date_start, date_end_mod)
    .filterBounds(greenland)
)
# print(test_image.getInfo())

# JAXA GCOM
jaxa_col = ee.ImageCollection('JAXA/GCOM-C/L3/LAND/LST/V3').filterDate('2024-01-01', '2024-12-31').filterBounds(greenland)
jaxa_test = jaxa_col.first().select('LST_AVE').multiply(0.02).subtract(273.15).rename('JAXA_LST_C')
img_a = ee.Image('JAXA/GCOM-C/L3/LAND/LST/V3/20240102A').select('LST_AVE').multiply(0.02).subtract(273.15).rename('JAXA_LST_C')
img_d = ee.Image('JAXA/GCOM-C/L3/LAND/LST/V3/20240102D').select('LST_AVE').multiply(0.02).subtract(273.15).rename('JAXA_LST_C')
jaxa_viz = {'bands': ['JAXA_LST_C'], 'min': -40, 'max': 20, 'palette': 'coolwarm'}

# print(jaxa_col.size().getInfo())
# jaxa_col


# testmap = geemap.Map()
# # testmap.addLayer(jaxa_test, jaxa_viz, 'JAXA GCOM')
# testmap.addLayer(img_a, jaxa_viz, 'JAXA GCOM AM')
# testmap.addLayer(img_d, jaxa_viz, 'JAXA GCOM PM')
# # testmap.addLayer(test_image2, {'bands': ['LST_Day_1km_C'], 'min': -30, 'max': 30}, 'MODIS LST Day')
# testmap.centerObject(greenland, 4)
# testmap.addLayerControl()
# testmap


False
724


Map(center=[72.70302525720832, -41.7868898523626], controls=(WidgetControl(options=['position', 'transparent_b…

### Cloud Masking

In [21]:
# Mask data based on quality flags

def bitwiseExtract(input, fromBit, toBit):
    maskSize = ee.Number(1).add(toBit).subtract(fromBit)
    mask = ee.Number(1).leftShift(maskSize).subtract(1)
    return input.rightShift(fromBit).bitwiseAnd(mask)

def maskQualityDaytime(image):
    qa = image.select('QC_Day')

    # Bits 0-"1": Mandatory QA flags
    # "0": LST produced, good quality, not necessary to examine more detailed QA
    # "1": LST produced, other quality, recommend examination of more detailed QA
    # "2": LST not produced due to cloud effects
    # "3": LST not produced primarily due to reasons other than cloud
    # bits01Mask = bitwiseExtract(qa, 0, 1).eq(0)
    bits01Mask = bitwiseExtract(qa, 0, 1).lte(1); 
    # Bits 2-"3": Data quality flag
    # "0": Good data quality
    # "1": Other quality data
    # "2": TBD
    # "3": TBD
    bits23Mask = bitwiseExtract(qa, 2, 3).eq(0)
    # Bits 4-"5": Emissivity error flag
    # "0": Average emissivity error <= 0.01
    # "1": 0.01 < Average emissivity error <= 0.02
    # "2": 0.02 < Average emissivity error <= 0.04
    # "3": Average emissivity error > 0.04
    bits45Mask = bitwiseExtract(qa, 4, 5).eq(0)
    # Bit 6-"7": LST error flag
    # "0": Average LST error <= 1K
    # "1": Average LST error <= 2K
    # "2": Average LST error <= 3K
    # "3": Average LST error > 3K
    bit6Mask = bitwiseExtract(qa, 6, 7).lte(1)

    mask = bits01Mask.And(bits23Mask).And(bits45Mask).And(bit6Mask)

    return image.updateMask(mask)

def maskQualityNighttime(image):
    qa = image.select('QC_Night')

    # Bits 0-"1": Mandatory QA flags
    # "0": LST produced, good quality, not necessary to examine more detailed QA
    # "1": LST produced, other quality, recommend examination of more detailed QA
    # "2": LST not produced due to cloud effects
    # "3": LST not produced primarily due to reasons other than cloud
    # bits01Mask = bitwiseExtract(qa, 0, 1).eq(0)
    bits01Mask = bitwiseExtract(qa, 0, 1).lte(1); 
    # Bits 2-"3": Data quality flag
    # "0": Good data quality
    # "1": Other quality data
    # "2": TBD
    # "3": TBD
    bits23Mask = bitwiseExtract(qa, 2, 3).eq(0)
    # Bits 4-"5": Emissivity error flag
    # "0": Average emissivity error <= 0.01
    # "1": 0.01 < Average emissivity error <= 0.02
    # "2": 0.02 < Average emissivity error <= 0.04
    # "3": Average emissivity error > 0.04
    bits45Mask = bitwiseExtract(qa, 4, 5).eq(0)
    # Bit 6-"7": LST error flag
    # "0": Average LST error <= 1K
    # "1": Average LST error <= 2K
    # "2": Average LST error <= 3K
    # "3": Average LST error > 3K
    bit6Mask = bitwiseExtract(qa, 6, 7).lte(1)

    mask = bits01Mask.And(bits23Mask).And(bits45Mask).And(bit6Mask)

    return image.updateMask(mask)


def maskJaxa(image):
    '''Function to filter JAXA GCOM-C LST data based on quality flag.'''
    qa = image.select('LST_QA_flag')
    #0: water (land fraction = 0%)
    #1: mostly water (0% < land fraction < 50%)
    #2: mostly coastal (50% < land fraction < 100%) - included
    #3: land (land fraction = 100%) - included
    mask = qa.gt(1)
    return image.updateMask(mask)


### Image collection
- band selection
- Temperature conversion
- load modis collections terra + aqua 
- filter Date
- filter greenland 
- apply QA mask
- apply temperature conversion



In [31]:
# Functions for band selection and conversion

# MODIS
def lst_mod_day(image):
    'Terra Day band selection and conversion'
    lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('MOD_LST_Day')
    qa_day = image.select('QC_Day').rename('MOD_QA_Day')
    return image.addBands(lst_day).addBands(qa_day)

def lst_mod_night(image):
    'Terra Night band selection and conversion'
    lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('MOD_LST_Night')
    qa_night = image.select('QC_Night').rename('MOD_QA_Night')
    return image.addBands(lst_night).addBands(qa_night)

def lst_myd_day(image):
    'Aqua Day band selection and conversion'
    lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('MYD_LST_Day')
    qa_day = image.select('QC_Day').rename('MYD_QA_Day')
    return image.addBands(lst_day).addBands(qa_day)

def lst_myd_night(image):
    'Aqua Night band selection and conversion'
    lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('MYD_LST_Night')
    qa_night = image.select('QC_Night').rename('MYD_QA_Night')
    return image.addBands(lst_night).addBands(qa_night)


# JAXA
def lst_jaxa(image):
    'JAXA GCOM-C band selection and conversion'
    lst_ave = image.select('LST_AVE').multiply(0.02).subtract(273.15).rename('JAXA_LST_AVE')
    qa_flag = image.select('LST_QA_flag')
    return image.addBands(lst_ave).addBands(qa_flag)



# Load MODIS Terra and Aqua data, apply quality control and conversion functions
MOD11A1Daytime = (
    ee.ImageCollection('MODIS/061/MOD11A1')
    .select(['LST_Day_1km', 'QC_Day'])
    .filterDate(date_start, date_end_mod)
    .filterBounds(greenland)
    .map(maskQualityDaytime)
    .map(lst_mod_day)
)

MOD11A1Nighttime = (
    ee.ImageCollection('MODIS/061/MOD11A1')
    .select(['LST_Night_1km', 'QC_Night'])
    .filterDate(date_start, date_end_mod)
    .filterBounds(greenland)
    .map(maskQualityNighttime)
    .map(lst_mod_night)
)

MYD11A1Daytime = (
    ee.ImageCollection('MODIS/061/MYD11A1')
    .select(['LST_Day_1km', 'QC_Day'])
    .filterDate(date_start, date_end_myd)
    .filterBounds(greenland)
    .map(maskQualityDaytime)
    .map(lst_myd_day)
)

MYD11A1Nighttime = (
    ee.ImageCollection('MODIS/061/MYD11A1')
    .select(['LST_Night_1km', 'QC_Night'])
    .filterDate(date_start, date_end_myd)
    .filterBounds(greenland)
    .map(maskJaxa)
    .map(lst_jaxa)
)

JAXA_A = (
    ee.ImageCollection('JAXA/GCOM-C/L3/LAND/LST/V3')
    .select(['LST_AVE', 'LST_QA_flag'])
    .filter(ee.Filter.eq('SATELLITE_DIRECTION', 'A')) # Filter for ascending (AM) overpasses
    .filter(ee.Filter.eq('PROCESSING_RESULT', 'Good')) # *************** NEW OBS! CHECK IF CORRECT
    .filterDate('2021-11-29', '2024-12-31')
    .filterBounds(greenland)
    .map(maskJaxa)
    .map(lst_jaxa)
)

JAXA_D = (
    ee.ImageCollection('JAXA/GCOM-C/L3/LAND/LST/V3')
    .select(['LST_AVE', 'LST_QA_flag'])
    .filter(ee.Filter.eq('SATELLITE_DIRECTION', 'D')) # Filter for descending (PM) overpasses
    .filter(ee.Filter.eq('PROCESSING_RESULT', 'Good')) # *************** NEW OBS! CHECK IF CORRECT
    .filterDate('2021-11-29', '2024-12-31')
    .filterBounds(greenland)
    .map(maskJaxa)
    .map(lst_jaxa)
)

# print('Images in collection Jaxa A', JAXA_A.size().getInfo())     - reveals 1125
# print('Images in collection Jaxa B', JAXA_D.size().getInfo())     - reveals 1125

# # Create a stack and select only images that cover aws points from stack
# modis_aws = (
#     MOD11A1Daytime.merge(MOD11A1Nighttime).merge(MYD11A1Daytime).merge(MYD11A1Nighttime)
#     .filterBounds(awsPoints)
# )

# # 2.3 Load ERA5 Land data (surface net solar radiation) and convert to daily average
# ERA5Land = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY') \
# .select('surface_net_solar_radiation', 'skin_temperature') \
# .filterDate(startDate, endDate) \
# .filterBounds(greenland)

# def func_hkp(image):
#     return image.updateMask(greenlandmask) \
# .map(func_hkp)




### Extraction of MODIS LST at AWS locations
- Create buffer of 1000m around aws points
- Create function for extraction of zonal statistics, including meta data and params. 
- Extract 

In [ ]:
# Compute zonal Statistics and set defaults

# Create a buffer function 
def bufferPoints(radius, bounds):
    def func_lws(pt):
        pt = ee.Feature(pt)
        return pt.buffer(radius).bounds() if bounds else pt.buffer(radius)
    return func_lws

points_buffered = awsPoints.map(bufferPoints(1000, True))
print(points_buffered.getInfo())


# Create a function computes zonal statistics of an ImageCollection over a FeatureCollection, including all
# necessary meta data. 
def zonalStats(ic, fc, params=None):
    'Compute zonal statistics of an ImageCollection over a FeatureCollection.'

    ## Initialize necessary parameters
    _params = {
        'reducer': ee.Reducer.mean(),
        'scale': None,
        'crs': None,
        'bands': None,
        'bandsRename': None,
        'imgProps': None,
        'imgPropsRename': None,
        'datetimeName': 'datetime',
        'datetimeFormat': 'YYYY-MM-dd HH:mm:ss'
    }

    ## Replace default params with (manually) provided params.
    if params:
        for k, v in params.items():
            _params[k] = v if v is not None else _params[k]

    ## Derive defaults from a representative image (first in collection)
    img_rep = ic.first()
    non_system_props = ee.Feature(None).copyProperties(img_rep).propertyNames()
    if _params['bands'] is None:
        _params['bands'] = img_rep.bandNames()
    if _params['bandsRename'] is None:
        _params['bandsRename'] = _params['bands']
    if _params['imgProps'] is None:
        _params['imgProps'] = non_system_props
    if _params['imgPropsRename'] is None:
        _params['imgPropsRename'] = _params['imgProps']

    ## Map over the ImageCollection
    def per_image(img):
        img = (ee.Image(img)
               .select(_params['bands'], _params['bandsRename'])
               .set(_params['datetimeName'],
                    img.date().format(_params['datetimeFormat']))
               .set('timestamp', img.get('system:time_start')))

        props_from = ee.List(_params['imgProps']) \
                       .cat(ee.List([_params['datetimeName'], 'timestamp']))
        props_to = ee.List(_params['imgPropsRename']) \
                       .cat(ee.List([_params['datetimeName'], 'timestamp']))
        img_props = img.toDictionary(props_from).rename(props_from, props_to)

        fc_sub = fc.filterBounds(img.geometry())

        def add_props(f):
            return ee.Feature(f).set(img_props).set('sampleID', f.get('name'))

        return img.reduceRegions(
            collection=fc_sub,
            reducer=_params['reducer'],
            scale=_params['scale'],
            crs=_params['crs']
        ).map(add_props)

    results = ic.map(per_image).flatten() \
                .filter(ee.Filter.notNull(_params['bandsRename'])) ### TRY WITHOUT THIS

    return results


##########
##########
##########

# Call zonalStats on all datasets separately. Stack threw errors (not all bands were found in every image) and loops are not recommended in Earth Engine.
results_mod_day = zonalStats(MOD11A1Daytime, points_buffered, {
        "reducer": ee.Reducer.mean(),
        "scale": 1000,
        "crs": 'EPSG:3413',
        "imgProps": ['system:index', 'system:time_start'],
        "datetimeName": 'date',
        "datetimeFormat": 'YYYY-MM-dd'
})

results_mod_night = zonalStats(MOD11A1Nighttime, points_buffered, {
        "reducer": ee.Reducer.mean(),
        "scale": 1000,
        "crs": 'EPSG:3413',
        "imgProps": ['system:index', 'system:time_start'],
        "datetimeName": 'date',
        "datetimeFormat": 'YYYY-MM-dd'
})

results_myd_day = zonalStats(MYD11A1Daytime, points_buffered, {
        "reducer": ee.Reducer.mean(),
        "scale": 1000,
        "crs": 'EPSG:3413',
        "imgProps": ['system:index', 'system:time_start'],
        "datetimeName": 'date',
        "datetimeFormat": 'YYYY-MM-dd'
})

results_myd_night = zonalStats(MYD11A1Nighttime, points_buffered, {
        "reducer": ee.Reducer.mean(),
        "scale": 1000,
        "crs": 'EPSG:3413',
        "imgProps": ['system:index', 'system:time_start'],
        "datetimeName": 'date',
        "datetimeFormat": 'YYYY-MM-dd'
})

results_jaxa_a = zonalStats(JAXA_A, points_buffered, {
        "reducer": ee.Reducer.mean(),
        "scale": 1000,
        "crs": 'EPSG:3413',
        "imgProps": ['system:index', 'system:time_start', 'SATELLITE_DIRECTION', 'PROCESSING_RESULT'],
        "datetimeName": 'date',
        "datetimeFormat": 'YYYY-MM-dd'
})

results_jaxa_d = zonalStats(JAXA_D, points_buffered, {
        "reducer": ee.Reducer.mean(),
        "scale": 1000,
        "crs": 'EPSG:3413',
        "imgProps": ['system:index', 'system:time_start', 'SATELLITE_DIRECTION', 'PROCESSING_RESULT'],
        "datetimeName": 'date',
        "datetimeFormat": 'YYYY-MM-dd'
})

print(results_jaxa_a.size().getInfo()) # Without quality filter: A: 547
print(results_jaxa_d.size().getInfo()) # Without quality filter: D: 982


{'type': 'FeatureCollection', 'columns': {'id': 'String', 'system:index': 'String'}, 'features': [{'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-51.39243991114472, 64.11348689092756], [-51.35147407304052, 64.11348689092756], [-51.35147407304052, 64.1314811657562], [-51.39243991114472, 64.1314811657562], [-51.39243991114472, 64.11348689092756]]]}, 'id': '0', 'properties': {'id': 'Kobbefjord_M500'}}, {'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-53.539325252269514, 69.24449122410975], [-53.488851664591806, 69.24449122410975], [-53.488851664591806, 69.26248549925229], [-53.539325252269514, 69.26248549925229], [-53.539325252269514, 69.24449122410975]]]}, 'id': '1', 'properties': {'id': 'Disko_AWS2'}}, {'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-20.596518964580074, 74.45649683882466], [-20.529759510122446, 74.45649683882466], [-20.529759510122446, 74.474491114

### Export

In [34]:
# Export to Drive (CSV)
# task = ee.batch.Export.table.toDrive(
#     collection=results_mod_day,
#     description='GEM_2003_AWS_MOD_DAY_LST_bit6lte0',
#     fileFormat='CSV',
#     folder='gee/2003'
# )
# task.start()

# task2 = ee.batch.Export.table.toDrive(
#     collection=results_mod_night,
#     description='GEM_2003_AWS_MOD_NIGHT_LST_bit6lte1',
#     fileFormat='CSV',
#     folder='gee/2003'
# )
# task2.start()

# task3 = ee.batch.Export.table.toDrive(
#     collection=results_myd_day,
#     description='GEM_2003_AWS_MYD_DAY_LST_bit6lte0',
#     fileFormat='CSV',
#     folder='gee/2003'
# )
# task3.start()

# task4 = ee.batch.Export.table.toDrive(
#     collection=results_myd_night,
#     description='GEM_2003_AWS_MYD_NIGHT_LST_bit6lte1',
#     fileFormat='CSV',
#     folder='gee/2003'
# )
# task4.start()

task5 = ee.batch.Export.table.toDrive(
    collection=results_jaxa_a,
    description='GEM_2024_AWS_JAXA_A_LST',
    fileFormat='CSV',
    folder='gee/2024'
)
task5.start()

task6 = ee.batch.Export.table.toDrive(
    collection=results_jaxa_d,
    description='GEM_2024_AWS_JAXA_D_LST',
    fileFormat='CSV',
    folder='gee/2024'
)
task6.start()

ee.batch.Task.list()


[<Task 5T47H4SXARPQELX6PCWHP7ZB EXPORT_FEATURES: GEM_2024_AWS_JAXA_D_LST (READY)>,
 <Task BONNKPUQLNU2CFPOFJGX2N73 EXPORT_FEATURES: GEM_2024_AWS_JAXA_A_LST (READY)>,
 <Task LR4B5FZWPAKUX6FGBHWPHSGV EXPORT_FEATURES: GEM_2024_AWS_JAXA_D_LST (COMPLETED)>,
 <Task CBZXXEME7HCIXAAVOC7JQMMM EXPORT_FEATURES: GEM_2024_AWS_JAXA_A_LST (COMPLETED)>,
 <Task IV5M5WFPPTS6RB5XU6WFMUDL EXPORT_FEATURES: GEM_2003_AWS_MYD_DAY_LST_bit6lte0 (COMPLETED)>,
 <Task 4YNJFUBDCTMKLKSPZH3XXJE5 EXPORT_FEATURES: GEM_2003_AWS_MOD_DAY_LST_bit6lte0 (COMPLETED)>,
 <Task CFPEAV4CTX5LNXGUDYJWK7DO EXPORT_FEATURES: GEM_2003_AWS_MYD_NIGHT_LST_bit6lte1 (COMPLETED)>,
 <Task 565327VUIXOT4WJEMZMZAQI3 EXPORT_FEATURES: GEM_2003_AWS_MYD_DAY_LST_bit6lte1 (COMPLETED)>,
 <Task GMAGZ4NR7SEHQXBJKGZ3Y4LB EXPORT_FEATURES: GEM_2003_AWS_MOD_NIGHT_LST_bit6lte1 (COMPLETED)>,
 <Task MRJYLOAMI2M5XOSMGRWTTSYD EXPORT_FEATURES: GEM_2003_AWS_MOD_DAY_LST_bit6lte1 (COMPLETED)>,
 <Task L76NYT6W4JNPGZHTFA63OOFJ EXPORT_FEATURES: GEM_2003_AWS_MYD_NIGHT_LS

In [16]:
map2 = geemap.Map()

maskedDay = maskQualityDaytime(test_image2)
maskedNight = maskQualityNighttime(test_image2)

vis_params_d = {'bands': ['LST_Day_1km_C'], 'min': -60, 'max': 30, 'palette': 'coolwarm'}
vis_params_n = {'bands': ['LST_Night_1km_C'], 'min': -60, 'max': 30, 'palette': 'coolwarm'}


# map2.addLayer(masked, {"bands": ['LST_Day_1km'], "min": 13000, "max": 16500, "palette": ['blue', 'green', 'red']}, 'Masked LST Day 1km')
# map2.addLayer(maskedDay, vis_params_d, 'masked LST Day 1km')
# map2.addLayer(test_image2, vis_params_d, 'LST Day 1km')
# map2.addLayer(maskedNight, vis_params_n, 'masked LST Night 1km')
# map2.addLayer(test_image2, vis_params_n, 'LST Night 1km')
map2.addLayer(test_image2, vis_params_d, 'LST Day 1km')
map2.addLayer(awsPoints, {"color": 'red'}, 'AWS Points')
map2.centerObject(awsPoints, 4)
map2


Map(center=[72.03449787370987, -36.3038245414739], controls=(WidgetControl(options=['position', 'transparent_b…